In [ ]:
!pip install scikit-learn

In [ ]:
# ============================================================================
# AITA Social Situation Assessment -- Kaggle Benchmark Task
# ============================================================================
import kaggle_benchmarks as kbench
from dataclasses import dataclass
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import os
import re
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/nird96/aita-reddit-posts"
CSV_FILENAME = "aita_verdicts_unique_6000.csv"

NUM_SAMPLES = 250
RANDOM_SEED = 42

VERDICT_LABELS = ["user_is_fault", "user_ok"]

LABEL_SYNONYMS = {
    "user_is_fault": "user_is_fault",
    "fault": "user_is_fault",
    "at_fault": "user_is_fault",
    "at fault": "user_is_fault",
    "yta": "user_is_fault",
    "esh": "user_is_fault",
    "asshole": "user_is_fault",
    "the_asshole": "user_is_fault",
    "yes": "user_is_fault",
    "guilty": "user_is_fault",
    "wrong": "user_is_fault",
    "in_the_wrong": "user_is_fault",
    "in the wrong": "user_is_fault",
    "user_ok": "user_ok",
    "ok": "user_ok",
    "not_fault": "user_ok",
    "not_at_fault": "user_ok",
    "not at fault": "user_ok",
    "nta": "user_ok",
    "nah": "user_ok",
    "not_the_asshole": "user_ok",
    "not the asshole": "user_ok",
    "no": "user_ok",
    "innocent": "user_ok",
    "not_wrong": "user_ok",
    "not wrong": "user_ok",
    "justified": "user_ok",
}


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_and_prepare_data():
    """Load the AITA verdicts CSV and prepare binary labels."""
    csv_path = os.path.join(DATASET_ROOT, CSV_FILENAME)

    if not os.path.exists(csv_path):
        for dirpath, dirnames, filenames in os.walk(DATASET_ROOT):
            for fname in filenames:
                if fname.lower() == CSV_FILENAME.lower():
                    csv_path = os.path.join(dirpath, fname)
                    break

    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"Cannot find {CSV_FILENAME} under {DATASET_ROOT}"
        )

    df = pd.read_csv(csv_path)

    if "full post" in df.columns:
        df = df.rename(columns={"full post": "fullpost"})

    df["verdict_raw"] = df["verdict"].astype(str).str.strip().str.lower()

    def map_verdict(v):
        if v == "user_is_fault":
            return "user_is_fault"
        if v == "user_ok":
            return "user_ok"
        if v in ("yta", "esh"):
            return "user_is_fault"
        if v in ("nta", "nah"):
            return "user_ok"
        return None

    df["binary_verdict"] = df["verdict_raw"].apply(map_verdict)
    df = df.dropna(subset=["binary_verdict"]).reset_index(drop=True)

    df["fullpost"] = df["fullpost"].astype(str).str.strip()
    df["post"] = df["post"].astype(str).str.strip()
    df["title"] = df["title"].astype(str).str.strip()

    df["best_text"] = df.apply(
        lambda r: r["fullpost"] if len(str(r["fullpost"])) > 50
        else str(r["post"]), axis=1
    )
    df = df[df["best_text"].str.len() > 50].reset_index(drop=True)

    if len(df) == 0:
        raise ValueError("No posts remain after filtering.")

    return df


def select_stratified_samples(df, n=NUM_SAMPLES):
    """Select N samples ensuring both verdict classes are represented."""
    if len(df) == 0:
        raise ValueError("DataFrame is empty.")

    actual_n = min(n, len(df))

    fault_pool = df[df["binary_verdict"] == "user_is_fault"]
    ok_pool = df[df["binary_verdict"] == "user_ok"]

    if len(fault_pool) == 0:
        return ok_pool.sample(
            n=min(actual_n, len(ok_pool)), random_state=RANDOM_SEED
        ).reset_index(drop=True)
    if len(ok_pool) == 0:
        return fault_pool.sample(
            n=min(actual_n, len(fault_pool)), random_state=RANDOM_SEED
        ).reset_index(drop=True)

    n_fault = max(1, min(len(fault_pool), actual_n // 2))
    n_ok = max(1, min(len(ok_pool), actual_n - n_fault))
    n_fault = min(len(fault_pool), actual_n - n_ok)

    fault_selected = fault_pool.sample(n=n_fault, random_state=RANDOM_SEED)
    ok_selected = ok_pool.sample(n=n_ok, random_state=RANDOM_SEED)

    return pd.concat(
        [fault_selected, ok_selected]
    ).sort_index().reset_index(drop=True)


def normalize_verdict(raw):
    """Normalize LLM output to a canonical verdict label."""
    cleaned = raw.strip().lower()
    cleaned = re.sub(r"[^a-z_ ]", "", cleaned)

    if cleaned in VERDICT_LABELS:
        return cleaned
    if cleaned in LABEL_SYNONYMS:
        return LABEL_SYNONYMS[cleaned]

    not_fault_patterns = ["not.*fault", "not.*wrong", "not.*asshole",
                          "user.*ok", "nta", "nah", "justified", "innocent"]
    for pat in not_fault_patterns:
        if re.search(pat, cleaned):
            return "user_ok"

    fault_patterns = ["fault", "wrong", "asshole", "yta", "esh", "guilty"]
    for pat in fault_patterns:
        if re.search(pat, cleaned):
            return "user_is_fault"

    return "INVALID"


def extract_verdict_from_response(raw_response):
    """Extract verdict from raw LLM response using multiple strategies."""
    if not raw_response or not raw_response.strip():
        return ""

    text = raw_response.strip()

    # Strategy 1: Try to parse as JSON directly
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "verdict" in parsed:
            return parsed["verdict"]
    except (json.JSONDecodeError, TypeError):
        pass

    # Strategy 2: Find JSON object in the response text
    json_match = re.search(r'\{[^{}]*"verdict"\s*:\s*"([^"]+)"[^{}]*\}', text)
    if json_match:
        return json_match.group(1)

    # Strategy 3: Look for verdict keywords directly in the text
    text_lower = text.lower()

    # Check "not at fault" / "user_ok" patterns first (before "fault" patterns)
    not_fault_indicators = ["user_ok", "not at fault", "not the asshole",
                            "nta", "nah", "justified", "not wrong",
                            "not at_fault", "not_the_asshole"]
    for indicator in not_fault_indicators:
        if indicator in text_lower:
            return "user_ok"

    fault_indicators = ["user_is_fault", "at fault", "the asshole",
                        "yta", "esh", "guilty", "in the wrong",
                        "at_fault", "the_asshole"]
    for indicator in fault_indicators:
        if indicator in text_lower:
            return "user_is_fault"

    return text


def build_prompt(title, post, fullpost, best_text=""):
    """Build the social situation assessment prompt."""
    main_text = best_text if best_text else (
        fullpost if len(fullpost) > len(post) else post
    )

    MAX_POST_CHARS = 4000
    if len(main_text) > MAX_POST_CHARS:
        main_text = main_text[:MAX_POST_CHARS] + "... [truncated]"

    return (
        "You are an expert in social dynamics, interpersonal conflict, and "
        "ethical reasoning.\n\n"
        "Read the following Reddit post from r/AmItheAsshole. In this subreddit, "
        "users describe a social conflict and ask the community to judge whether "
        "they acted wrongly.\n\n"
        "Your job is to assess the social situation and determine: Is the poster "
        "at fault, or are they in the clear?\n\n"
        f"**Title:** {title}\n\n"
        f"**Post:**\n{main_text}\n\n"
        "Consider:\n"
        "- Power dynamics between the people involved\n"
        "- Whether any social norms or boundaries were violated\n"
        "- The level of tension and who is causing it\n"
        "- Whether the poster's actions were reasonable given the context\n"
        "- Any unwritten social rules at play\n\n"
        "Respond with ONLY a JSON object and nothing else. No explanation, "
        "no preamble, no markdown formatting. Just the raw JSON:\n"
        '{"verdict": "user_is_fault"} if the poster acted wrongly, was '
        "unreasonable, or shares significant blame\n"
        '{"verdict": "user_ok"} if the poster was justified, reasonable, '
        "or not at fault\n"
    )


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="aita_social_situation_assessment",
    description=(
        "Social situation assessment on AITA Reddit posts. "
        "The model receives a post describing an interpersonal conflict "
        "and must determine whether the poster is at fault (user_is_fault) "
        "or justified (user_ok). Scored by F1-macro."
    ),
)
def aita_social_situation_assessment(llm) -> float:
    """
    Evaluate an LLM's ability to read social dynamics and render verdicts
    on interpersonal conflicts. Returns F1-macro between 0.0 and 1.0.
    """

    df = load_and_prepare_data()
    samples = select_stratified_samples(df, n=NUM_SAMPLES)

    y_true = []
    y_pred = []
    results_log = []
    total = len(samples)

    for i, (idx, row) in enumerate(samples.iterrows()):
        gt_verdict = row["binary_verdict"]
        title = row["title"]
        post = row["post"]
        fullpost = row["fullpost"]
        best_text = row.get("best_text", "")

        prompt_text = build_prompt(title, post, fullpost, best_text)

        raw_verdict = ""
        with kbench.chats.new(f"sample_{i}"):
            try:
                raw_response = llm.prompt(prompt_text)
                raw_verdict = extract_verdict_from_response(
                    raw_response if raw_response else ""
                )
            except Exception as e:
                print(f"[Sample {i+1}] Error: {e}")
                raw_verdict = ""

        # If first attempt failed, retry once in a fresh chat
        if not raw_verdict or normalize_verdict(str(raw_verdict)) == "INVALID":
            with kbench.chats.new(f"sample_{i}_retry"):
                try:
                    raw_response = llm.prompt(prompt_text)
                    raw_verdict = extract_verdict_from_response(
                        raw_response if raw_response else ""
                    )
                except Exception as e:
                    print(f"[Sample {i+1} retry] Error: {e}")
                    raw_verdict = "user_ok"  # fallback default

        pred_verdict = normalize_verdict(str(raw_verdict))

        # Final fallback if still invalid
        if pred_verdict == "INVALID":
            pred_verdict = "user_ok"

        y_true.append(gt_verdict)
        y_pred.append(pred_verdict)

        print(f"[Sample {i+1}/{total}] pid={row['pid']} | "
              f"GT={gt_verdict} | Pred={pred_verdict} | "
              f"{'CORRECT' if pred_verdict == gt_verdict else 'WRONG'}")

        results_log.append({
            "sample": i + 1,
            "pid": row["pid"],
            "gt_verdict": gt_verdict,
            "pred_verdict": pred_verdict,
            "correct": pred_verdict == gt_verdict,
        })

    # Compute F1-macro
    f1_macro = f1_score(
        y_true, y_pred,
        average="macro", labels=VERDICT_LABELS, zero_division=0,
    )

    # Print summary
    print("\n" + "=" * 60)
    print("RESULTS SUMMARY")
    print("=" * 60)

    results_df = pd.DataFrame(results_log)
    print(results_df.to_string(index=False))

    correct_count = sum(r["correct"] for r in results_log)
    invalid_labels = sum(1 for p in y_pred if p == "INVALID")

    print(f"\nAccuracy:    {correct_count}/{total} ({correct_count/total:.1%})")
    print(f"F1-macro:    {f1_macro:.4f}")
    print(f"Invalid:     {invalid_labels}/{total}")

    if total >= 4:
        print(f"\n{classification_report(y_true, y_pred, labels=VERDICT_LABELS, zero_division=0)}")

    print("=" * 60)

    # Assertions
    kbench.assertions.assert_true(
        0.0 <= f1_macro <= 1.0,
        expectation="F1-macro score should be between 0 and 1.",
    )

    kbench.assertions.assert_true(
        invalid_labels < total,
        expectation="At least one prediction should be a valid verdict label.",
    )

    kbench.assertions.assert_true(
        correct_count > 0,
        expectation="Model should correctly assess at least one sample.",
    )

    return f1_macro


# ============================================================================
# RUN THE TASK
# ============================================================================

aita_social_situation_assessment.run(kbench.llm)

In [ ]:
%choose aita_social_situation_assessment